# Gold — Fact Monthly Economics (Kimball Periodic Snapshot)
Joins Silver tables and unpivots into a narrow periodic snapshot fact table following Kimball conventions.

**Sources:** `silver.central_bank_indicators`, `silver.gdp_indicators`, `silver.statistics_monthly_indicators`\
**Dimensions:** `gold.dim_date`, `gold.dim_indicator`, `gold.dim_source`, `gold.dim_geography`\
**Grain:** One row per calendar month per economic indicator\
**GDP note:** Quarterly values are assigned to the first month of their quarter (Q1→Jan, Q2→Apr, Q3→Jul, Q4→Oct)

In [ ]:
df_cb = spark.read.table("silver.central_bank_indicators")
df_gdp = spark.read.table("silver.gdp_indicators")
df_monthly = spark.read.table("silver.statistics_monthly_indicators")

df_cb.createOrReplaceTempView("v_central_bank_indicators")
df_gdp.createOrReplaceTempView("v_gdp_indicators")
df_monthly.createOrReplaceTempView("v_monthly_indicators")

In [25]:
df_gold = spark.sql("""
    WITH monthly_policy AS (
        SELECT 
            YEAR(date) AS year,
            MONTH(date) AS month,
            ROUND(value, 4) AS value,
            'policy_rate' AS metric
        FROM (
            SELECT
                date,
                value,
                ROW_NUMBER() OVER (PARTITION BY YEAR(date), MONTH(date) ORDER BY date DESC) AS rn
            FROM v_central_bank_indicators WHERE metric = 'policy_rate'
        ) WHERE rn = 1
    ),
    monthly_cpi AS (
        SELECT
            YEAR(date) AS year,
            MONTH(date) AS month,
            ROUND(value, 4) AS value,
            'cpi' AS metric
        FROM v_central_bank_indicators
        WHERE metric = 'cpi'
    ),
    monthly_iskeur AS (
        SELECT
            YEAR(date) AS year,
            MONTH(date) AS month,
            ROUND(AVG(value), 4) AS value,
            'iskeur' AS metric
        FROM v_central_bank_indicators
        WHERE metric = 'iskeur'
        GROUP BY YEAR(date), MONTH(date)
    ),
    monthly_gdp AS (
        SELECT
            year,
            ((q - 1) * 3 + 1) AS month,
            ROUND(value, 2) AS value,
            'gdp_yoy_growth' AS metric
        FROM v_gdp_indicators
    ),
    monthly_stats AS (
        SELECT
            year,
            month,
            ROUND(value, 2) AS value,
            metric
        FROM v_monthly_indicators
    ),
    all_indicators AS (
        SELECT year, month, value, metric FROM monthly_policy
        UNION ALL
        SELECT year, month, value, metric FROM monthly_cpi
        UNION ALL
        SELECT year, month, value, metric FROM monthly_iskeur
        UNION ALL
        SELECT year, month, value, metric FROM monthly_gdp
        UNION ALL
        SELECT year, month, value, metric FROM monthly_stats
    )
    SELECT
        (a.year * 100 + a.month) AS month_key,
        (a.year * 10000 + a.month * 100 + 1) AS date_key,
        i.indicator_key,
        i.source_key,
        1 AS geography_key,
        a.value,
        FALSE AS is_estimated,
        CURRENT_TIMESTAMP() AS row_created_at,
        CURRENT_TIMESTAMP() AS row_updated_at
    FROM all_indicators a
    JOIN gold.dim_indicator i ON a.metric = i.indicator_code
    WHERE a.value IS NOT NULL
""")

StatementMeta(, d77caa68-c7c6-41c8-9249-4b6e7224be56, 27, Finished, Available, Finished, False)

In [26]:
df_gold.createOrReplaceTempView("v_fact_monthly_economics")

if not spark.catalog.tableExists("gold.fact_monthly_economics"):
    df_gold.write.format("delta").saveAsTable("gold.fact_monthly_economics")
else:
    spark.sql("""
        MERGE INTO gold.fact_monthly_economics AS target
        USING v_fact_monthly_economics AS source
        ON target.month_key = source.month_key
        AND target.indicator_key = source.indicator_key
        WHEN MATCHED THEN UPDATE SET
            target.date_key       = source.date_key,
            target.source_key     = source.source_key,
            target.value          = source.value,
            target.row_updated_at = source.row_updated_at
        WHEN NOT MATCHED THEN INSERT (
            month_key, date_key, indicator_key, source_key, geography_key,
            value, is_estimated, row_created_at, row_updated_at
        )
        VALUES (
            source.month_key, source.date_key, source.indicator_key, source.source_key,
            source.geography_key, source.value, source.is_estimated,
            source.row_created_at, source.row_updated_at
        )
    """)

print(f"Gold table updated. Total rows: {spark.table('gold.fact_monthly_economics').count()}")

StatementMeta(, d77caa68-c7c6-41c8-9249-4b6e7224be56, 28, Finished, Available, Finished, False)

Saved to gold.fact_monthly_economics - 467 rows
